# LeetCode 3575 — Maximum Good Subtree Score bằng Python

- Link: https://leetcode.com/problems/maximum-good-subtree-score/
- Độ khó: **Hard**
- Chủ đề: **Bitmask các chữ số**, **quy hoạch động trên cây**, **knapsack trên tập con** (subset DP), **gộp nhỏ vào lớn**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def goodSubtreeSum(self, vals: List[int], par: List[int]) -> int:
```

## 1. Mục tiêu học

1. Mã hóa "tập chữ số xuất hiện trong một số" thành **bitmask 10 bit**; loại bỏ ngay các số có chữ số lặp (không bao giờ được chọn).
2. Nhận ra "tập con tốt" ⇔ các mask **đôi một rời nhau** ⇒ bài toán **đóng gói tập** (set packing) trên vũ trụ chỉ 10 phần tử.
3. Viết **knapsack theo mask**: `g[S]` = tổng lớn nhất dùng các chữ số trong `S`; thêm một vật `(m, v)` bằng cách duyệt các tập con của phần bù.
4. Áp dụng cho **mọi cây con**: DP gộp con bằng tích chập max-plus `O(3^10)` hoặc **gộp nhỏ vào lớn** chỉ thêm vật của cây con nhỏ.
5. Kiểm chứng bằng vét cạn mọi tập con trên cây nhỏ; nhớ lấy **modulo** ở cuối.

## 2. Đề bài & ví dụ

Cây gốc `0`, `n` đỉnh, cha của `i` là `par[i]`, giá trị `vals[i]`.

Một **tập con các đỉnh** trong cây con của `u` gọi là **tốt** nếu mỗi chữ số `0..9` xuất hiện **tối đa một lần** khi viết thập phân **tất cả** các giá trị được chọn. **Điểm** của tập = tổng giá trị.

`maxScore[u]` = điểm lớn nhất của một tập tốt nằm trong cây con của `u`. Trả về `Σ maxScore[u]` modulo `10^9 + 7`.

**Ví dụ 1:** `vals=[2,3], par=[-1,0]` → `8` (`maxScore = [5, 3]`).

**Ví dụ 2:** `vals=[1,5,2], par=[-1,0,0]` → `15` (`[8, 5, 2]`).

**Ví dụ 3:** `vals=[34,1,2], par=[-1,0,1]` → `42` (`[37, 3, 2]`).

**Ví dụ 4:** `vals=[3,22,5], par=[-1,0,1]` → `18` (`22` có chữ số 2 lặp nên không bao giờ chọn được; `[8, 5, 5]`).

**Ràng buộc:** `1 ≤ n ≤ 500`, `1 ≤ vals[i] ≤ 10^9`.

In [1]:
from typing import List
MOD = 10**9 + 7
FULL = (1 << 10) - 1
examples = [
    ([2, 3], [-1, 0], 8),
    ([1, 5, 2], [-1, 0, 0], 15),
    ([34, 1, 2], [-1, 0, 1], 42),
    ([3, 22, 5], [-1, 0, 1], 18),
]
for vals, par, exp in examples:
    print(vals, par, '-> kỳ vọng', exp)

[2, 3] [-1, 0] -> kỳ vọng 8
[1, 5, 2] [-1, 0, 0] -> kỳ vọng 15
[34, 1, 2] [-1, 0, 1] -> kỳ vọng 42
[3, 22, 5] [-1, 0, 1] -> kỳ vọng 18


## 3. Kiến thức nền

### 3.1. Mask chữ số của một số

Duyệt các chữ số của `str(v)`, bật bit tương ứng. Nếu một bit đã bật rồi ⇒ chữ số lặp ⇒ trả `-1` (số này không thể nằm trong bất kỳ tập tốt nào, vì chính nó đã vi phạm).

In [2]:
def digit_mask(v):
    m = 0
    for ch in str(v):
        b = 1 << (ord(ch) - 48)
        if m & b:
            return -1
        m |= b
    return m

for v in [34, 22, 1023456789, 1000, 7]:
    m = digit_mask(v)
    print(v, '->', m if m < 0 else format(m, '010b'))

34 -> 0000011000
22 -> -1
1023456789 -> 1111111111
1000 -> -1
7 -> 0010000000


### 3.2. Hai mask rời nhau

Hai số chọn chung được ⇔ không có chữ số chung ⇔ `m1 & m2 == 0`. Hợp các chữ số: `m1 | m2`.

In [3]:
a, b, c = digit_mask(34), digit_mask(1), digit_mask(13)
print('34 & 1 =', a & b, '-> chọn chung được' if a & b == 0 else '-> xung đột')
print('34 & 13 =', a & c, '-> chọn chung được' if a & c == 0 else '-> xung đột (chữ số 3)')

34 & 1 = 0 -> chọn chung được
34 & 13 = 8 -> xung đột (chữ số 3)


### 3.3. Duyệt mọi tập con của một mask

`sub = comp; while True: ...; if sub == 0: break; sub = (sub - 1) & comp` — duyệt cả tập rỗng. Tổng số cặp (mask, tập con) trên 10 bit là `3^10 = 59 049`.

In [4]:
comp = 0b1011
sub, out = comp, []
while True:
    out.append(format(sub, '04b'))
    if sub == 0:
        break
    sub = (sub - 1) & comp
print(out, '| 3^10 =', 3 ** 10)

['1011', '1010', '1001', '1000', '0011', '0010', '0001', '0000'] | 3^10 = 59049


### 3.4. Knapsack theo mask: thêm một vật

`g[S]` = tổng lớn nhất của các vật đã thêm, đôi một rời nhau, với hợp mask **nằm trong** `S` (khởi tạo `g = [0]*1024`: chọn rỗng). Thêm vật `(m, v)`:

`g[m | sub] = max(g[m | sub], v + g[sub])` với mọi `sub ⊆ ~m`.

Vì `sub` không chứa bit nào của `m`, giá trị `g[sub]` dùng ở vế phải chưa bị cập nhật trong lượt này ⇒ mỗi vật dùng tối đa một lần (không cần duyệt ngược như knapsack 0/1 thông thường).

In [5]:
def add_item(g, m, v):
    comp = FULL ^ m
    sub = comp
    while True:
        x = v + g[sub]
        if x > g[m | sub]:
            g[m | sub] = x
        if sub == 0:
            break
        sub = (sub - 1) & comp

g = [0] * 1024
for v in [34, 1, 2, 13]:
    add_item(g, digit_mask(v), v)
print('Tốt nhất với mọi chữ số:', g[FULL])   # 34 + 1 + 2 = 37 hoặc 13 + 2 = 15 -> 37

Tốt nhất với mọi chữ số: 37


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Mỗi đỉnh hợp lệ là một "vật" `(mask, value)`. Câu hỏi cho đỉnh `u`: chọn các vật trong cây con của `u`, mask đôi một rời nhau, tổng giá trị lớn nhất.

Vũ trụ chữ số chỉ có 10 phần tử ⇒ một tập tốt có tối đa 10 đỉnh và trạng thái "đã dùng chữ số nào" chỉ có `2^10 = 1024` khả năng.

Tính tay ví dụ 4 (`vals = [3, 22, 5]`, đường thẳng `0 → 1 → 2`):

| đỉnh | mask | vật trong cây con | maxScore |
|---|---|---|---|
| 2 | {5} | 5 | 5 |
| 1 | `22` lặp ⇒ bỏ | 5 | 5 |
| 0 | {3} | 3, 5 | 8 |

Tổng `18`.

In [6]:
def items_of_subtree(vals, par, u):
    n = len(vals)
    children = [[] for _ in range(n)]
    for i in range(1, n):
        children[par[i]].append(i)
    st, out = [u], []
    while st:
        x = st.pop()
        m = digit_mask(vals[x])
        if m > 0:
            out.append((m, vals[x]))
        st.extend(children[x])
    return out

for u in range(3):
    print('cây con', u, '-> vật hợp lệ:', [(format(m, '010b'), v) for m, v in items_of_subtree([3, 22, 5], [-1, 0, 1], u)])

cây con 0 -> vật hợp lệ: [('0000001000', 3), ('0000100000', 5)]
cây con 1 -> vật hợp lệ: [('0000100000', 5)]
cây con 2 -> vật hợp lệ: [('0000100000', 5)]


## 5. Cách 1 — Brute Force: thử mọi tập con trong mỗi cây con

Với mỗi đỉnh `u`, liệt kê mọi tập con các vật trong cây con, kiểm tra các mask đôi một rời nhau, lấy tổng lớn nhất.

In [7]:
def good_brute(vals, par):
    n = len(vals)
    total = 0
    for u in range(n):
        items = items_of_subtree(vals, par, u)
        best = 0
        k = len(items)
        for pick in range(1 << k):
            used, s, ok = 0, 0, True
            for i in range(k):
                if pick >> i & 1:
                    m, v = items[i]
                    if used & m:
                        ok = False
                        break
                    used |= m
                    s += v
            if ok:
                best = max(best, s)
        total += best
    return total % MOD

for vals, par, exp in examples:
    print(good_brute(vals, par), '| kỳ vọng', exp)

8 | kỳ vọng 8
15 | kỳ vọng 15
42 | kỳ vọng 42
18 | kỳ vọng 18


**Độ phức tạp:** `O(n · 2^n · n)` — chỉ dùng với `n ≤ 14`.

**Nút thắt:** tập con của **đỉnh** thì mũ theo `n`, nhưng thông tin thật sự cần nhớ chỉ là **tập chữ số đã dùng** (1024 trạng thái).

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — DP theo mask chữ số.** Với một tập vật cố định, `g[S]` (knapsack theo mask, mục 3.4) cho đáp án `g[FULL]`. Thêm một vật tốn tối đa `2^(10 − |m|) ≤ 512` bước.

**Insight 2 — vật của cây con cha ⊇ vật của cây con con.** Nên `g` của cha có thể **kế thừa** `g` của một con rồi chỉ thêm vật còn thiếu.

**Insight 3 — kế thừa con lớn nhất (gộp nhỏ vào lớn).** Giữ nguyên mảng `g` của con có nhiều vật nhất, thêm vật của các con khác và của `u`. Mỗi vật bị "thêm lại" mỗi khi nó nằm trong một cây con **không** lớn nhất; số lần như vậy `O(log n)` ⇒ tổng `O(n log n · 512)`.

Quy tắc ngắn:

- **Nếu** `vals[u]` có chữ số lặp **thì** bỏ qua đỉnh `u` như một vật (nhưng vẫn tính `maxScore[u]`).
- **Nếu** `m1 & m2 ≠ 0` **thì** hai vật không đi cùng nhau.
- `maxScore[u] = g_u[FULL]`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Knapsack riêng cho từng cây con

Với mỗi `u`, gom vật trong cây con, chạy knapsack theo mask từ đầu. Đúng, nhưng tổng số vật xử lý là `Σ size(u)` — tới `O(n^2)` trên cây đường thẳng (125 000 lần thêm vật × ~512).

In [8]:
def good_per_subtree(vals, par):
    n = len(vals)
    total = 0
    for u in range(n):
        g = [0] * 1024
        for m, v in items_of_subtree(vals, par, u):
            add_item(g, m, v)
        total += g[FULL]
    return total % MOD

for vals, par, exp in examples:
    print(good_per_subtree(vals, par), '| kỳ vọng', exp)

8 | kỳ vọng 8
15 | kỳ vọng 15
42 | kỳ vọng 42
18 | kỳ vọng 18


Cải tiến nhỏ: hai vật cùng mask không bao giờ đi cùng nhau ⇒ chỉ giữ vật giá trị lớn nhất cho mỗi mask (tối đa 1023 vật). Không đổi bản chất `O(n^2)`.

### 7.2. Cách 3 — DP cây gộp con bằng tích chập max-plus trên tập con (theo gợi ý đề)

`dp[u][S]` = tổng lớn nhất với tập chữ số dùng **đúng bằng** `S` (−1 nếu không đạt được). Khởi tạo bằng `{∅: 0, m_u: vals[u]}`, sau đó gộp từng con `c`:

`new[a | b] = max(new[a | b], cur[a] + dp[c][b])` với `a & b = 0`.

Duyệt `b` qua tập con của `~a` ⇒ `O(3^10)` mỗi lần gộp ⇒ `O(n · 3^10)`.

In [9]:
def good_convolution(vals, par):
    n = len(vals)
    children = [[] for _ in range(n)]
    for i in range(1, n):
        children[par[i]].append(i)
    order = [0]
    for u in order:
        order.extend(children[u])
    dp = [None] * n
    total = 0
    for u in reversed(order):
        cur = [-1] * 1024
        cur[0] = 0
        m = digit_mask(vals[u])
        if m > 0:
            cur[m] = vals[u]
        for c in children[u]:
            d = dp[c]
            new = cur[:]
            for a in range(1024):
                if cur[a] < 0:
                    continue
                comp = FULL ^ a
                b = comp
                while b:
                    if d[b] >= 0 and cur[a] + d[b] > new[a | b]:
                        new[a | b] = cur[a] + d[b]
                    b = (b - 1) & comp
            cur = new
            dp[c] = None
        dp[u] = cur
        total += max(cur)
    return total % MOD

for vals, par, exp in examples:
    print(good_convolution(vals, par), '| kỳ vọng', exp)

8 | kỳ vọng 8
15 | kỳ vọng 15
42 | kỳ vọng 42
18 | kỳ vọng 18


**Độ phức tạp:** `O(n · 3^10) ≈ 500 · 59 049 ≈ 3·10^7` thao tác trong trường hợp xấu (mask dày đặc) — ổn với C++, chậm vài giây với Python. Nhờ bỏ qua `cur[a] < 0` nên thực tế thường nhanh hơn nhiều.

### 7.3. Cách 4 — Gộp nhỏ vào lớn + knapsack theo mask (lời giải chính)

Mỗi đỉnh giữ: mảng `g` (1024 số) và danh sách vật trong cây con. Kế thừa của con có **nhiều vật nhất**, thêm vật của các con còn lại và của chính `u`.

In [10]:
def good_small_to_large(vals, par):
    n = len(vals)
    children = [[] for _ in range(n)]
    for i in range(1, n):
        children[par[i]].append(i)
    order = [0]
    for u in order:
        order.extend(children[u])
    G, items = [None] * n, [None] * n
    total = 0
    for u in reversed(order):
        big = None
        for c in children[u]:
            if big is None or len(items[c]) > len(items[big]):
                big = c
        if big is None:
            g, it = [0] * 1024, []
        else:
            g, it = G[big], items[big]
        for c in children[u]:
            if c != big:
                for m, v in items[c]:
                    add_item(g, m, v)
                    it.append((m, v))
            G[c] = items[c] = None
        m = digit_mask(vals[u])
        if m > 0:
            add_item(g, m, vals[u])
            it.append((m, vals[u]))
        G[u], items[u] = g, it
        total += g[FULL]
    return total % MOD

for vals, par, exp in examples:
    print(good_small_to_large(vals, par), '| kỳ vọng', exp)

8 | kỳ vọng 8
15 | kỳ vọng 15
42 | kỳ vọng 42
18 | kỳ vọng 18


**Độ phức tạp:** mỗi vật được thêm `O(log n)` lần, mỗi lần `≤ 512` bước ⇒ `O(n log n · 2^10)`; Space `O(n · 2^10)` trong trường hợp xấu (nhiều mảng `g` sống cùng lúc), thực tế ít hơn vì giải phóng mảng của con.

## 8. TRACE từng bước

Cây 6 đỉnh:

```
          0 (12)
        /   \
   1 (30)    2 (45)
    /  \        \
 3 (7) 4 (11)   5 (6)
```

`11` có chữ số lặp ⇒ không phải vật. Trace cách 4: thứ tự xử lý, con được kế thừa, các vật được thêm và `maxScore` mỗi đỉnh.

In [11]:
def trace_small_to_large(vals, par):
    n = len(vals)
    children = [[] for _ in range(n)]
    for i in range(1, n):
        children[par[i]].append(i)
    order = [0]
    for u in order:
        order.extend(children[u])
    G, items = [None] * n, [None] * n
    scores = [0] * n
    for u in reversed(order):
        big = None
        for c in children[u]:
            if big is None or len(items[c]) > len(items[big]):
                big = c
        g, it = ([0] * 1024, []) if big is None else (G[big], items[big])
        added = []
        for c in children[u]:
            if c != big:
                for m, v in items[c]:
                    add_item(g, m, v); it.append((m, v)); added.append(v)
        m = digit_mask(vals[u])
        own = 'chữ số lặp -> bỏ' if m < 0 else f'thêm chính nó {vals[u]}'
        if m > 0:
            add_item(g, m, vals[u]); it.append((m, vals[u]))
        G[u], items[u] = g, it
        scores[u] = g[FULL]
        print(f"đỉnh {u} (vals={vals[u]:>2}): kế thừa con {big}, thêm từ con nhỏ {added}, {own}"
              f" -> vật = {[v for _, v in it]}, maxScore = {g[FULL]}")
    print('maxScore =', scores, '| tổng =', sum(scores))
    return sum(scores) % MOD

vals6, par6 = [12, 30, 45, 7, 11, 6], [-1, 0, 0, 1, 1, 2]
trace_small_to_large(vals6, par6)

đỉnh 5 (vals= 6): kế thừa con None, thêm từ con nhỏ [], thêm chính nó 6 -> vật = [6], maxScore = 6
đỉnh 4 (vals=11): kế thừa con None, thêm từ con nhỏ [], chữ số lặp -> bỏ -> vật = [], maxScore = 0
đỉnh 3 (vals= 7): kế thừa con None, thêm từ con nhỏ [], thêm chính nó 7 -> vật = [7], maxScore = 7
đỉnh 2 (vals=45): kế thừa con 5, thêm từ con nhỏ [], thêm chính nó 45 -> vật = [6, 45], maxScore = 51
đỉnh 1 (vals=30): kế thừa con 3, thêm từ con nhỏ [], thêm chính nó 30 -> vật = [7, 30], maxScore = 37
đỉnh 0 (vals=12): kế thừa con 1, thêm từ con nhỏ [6, 45], thêm chính nó 12 -> vật = [7, 30, 6, 45, 12], maxScore = 100
maxScore = [100, 37, 51, 7, 0, 6] | tổng = 201


201

Kiểm tra tay đỉnh 0: vật `{12, 30, 45, 7, 6}`. Một tập tốt lớn nhất: `45 + 30 + 7 + 6 = 88`? Chữ số: 4,5 / 3,0 / 7 / 6 — rời nhau ⇒ 88; thêm `12` thì đụng... `12` dùng 1,2 — không đụng! ⇒ `88 + 12 = 100`. Trace phải ra `100`.

In [12]:
print('vét cạn:', good_brute(vals6, par6))

vét cạn: 201


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def goodSubtreeSum(self, vals: List[int], par: List[int]) -> int:
        MOD = 10**9 + 7
        FULL = 1023
        n = len(vals)
        children = [[] for _ in range(n)]
        for i in range(1, n):
            children[par[i]].append(i)
        order = [0]
        for u in order:
            order.extend(children[u])

        def mask_of(v):
            m = 0
            for ch in str(v):
                b = 1 << (ord(ch) - 48)
                if m & b:
                    return -1
                m |= b
            return m

        def add(g, m, v):
            comp = FULL ^ m
            sub = comp
            while True:
                x = v + g[sub]
                if x > g[m | sub]:
                    g[m | sub] = x
                if sub == 0:
                    break
                sub = (sub - 1) & comp

        G, items = [None] * n, [None] * n
        total = 0
        for u in reversed(order):
            big = None
            for c in children[u]:
                if big is None or len(items[c]) > len(items[big]):
                    big = c
            if big is None:
                g, it = [0] * 1024, []
            else:
                g, it = G[big], items[big]
            for c in children[u]:
                if c != big:
                    for m, v in items[c]:
                        add(g, m, v)
                        it.append((m, v))
                G[c] = items[c] = None
            m = mask_of(vals[u])
            if m > 0:
                add(g, m, vals[u])
                it.append((m, vals[u]))
            G[u], items[u] = g, it
            total += g[FULL]
        return total % MOD

for vals, par, exp in examples:
    got = Solution().goodSubtreeSum(vals, par)
    print(got, 'OK' if got == exp else 'SAI')

8 OK
15 OK
42 OK
18 OK


## 10. Vì sao thuật toán đúng

1. **Đỉnh có chữ số lặp không bao giờ được chọn**: chính nó đã có một chữ số xuất hiện hai lần.
2. **Tập tốt ⇔ mask đôi một rời nhau** (với các đỉnh hợp lệ): mỗi chữ số xuất hiện tối đa một lần trên toàn bộ ⇔ không có hai số chung chữ số và không số nào tự lặp.
3. **Bất biến knapsack:** sau khi thêm tập vật `I`, `g[S]` = max tổng của tập con rời nhau của `I` có hợp mask `⊆ S`. Chứng minh quy nạp: thêm vật `(m, v)`, một tập tối ưu trong `S` hoặc không dùng vật mới (giá trị cũ `g[S]`), hoặc dùng nó cùng các vật còn lại nằm trong `S \ m` ⊆ phần bù của `m` (giá trị `v + g_cũ[S \ m]`). Cập nhật duyệt đúng các `S = m | sub`; với `S` không chứa `m`, vật không dùng được.
4. **Kế thừa đúng:** `g` của con lớn nhất phản ánh đúng tập vật của nó; thêm vật của các con khác và `u` cho đúng tập vật của cây con `u` (thứ tự thêm không quan trọng). Giá trị `maxScore[c]` đã được cộng trước khi mảng bị cha tái sử dụng.
5. **Modulo:** mỗi `maxScore[u] ≤ 9 876 543 210` (dùng hết 10 chữ số) nên tổng không lớn; lấy modulo ở cuối theo yêu cầu.

## 11. Phân tích Time / Space Complexity

Gọi `n ≤ 500`, `D = 10` chữ số, `2^D = 1024`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn tập con | `O(n^2 · 2^n)` | `O(n)` |
| 2. Knapsack riêng mỗi cây con | `O(Σ size(u) · 2^D) = O(n^2 · 2^D)` | `O(2^D)` |
| 3. Tích chập max-plus khi gộp con | `O(n · 3^D)` | `O(h · 2^D)` |
| 4. Gộp nhỏ vào lớn | `O(n log n · 2^D)` | `O(n · 2^D)` xấu nhất |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [14]:
impls = {
    'per_subtree': good_per_subtree,
    'convolution': good_convolution,
    'small_to_large': good_small_to_large,
    'submit': lambda v, p: Solution().goodSubtreeSum(v, p),
}
edge = [
    ([11], [-1], 0),                         # đỉnh duy nhất có chữ số lặp
    ([7], [-1], 7),
    ([1023456789], [-1], 1023456789),        # dùng đủ 10 chữ số
    ([5, 5, 5], [-1, 0, 0], 5 + 5 + 5),      # cùng mask: mỗi cây con chỉ chọn được một số 5
    ([98765432, 1], [-1, 0], (98765432 + 1) + 1),   # gốc dùng 9 chữ số + số 1
    ([987654321, 1], [-1, 0], 987654321 + 1),       # 987654321 đã chứa chữ số 1 -> xung đột
]
for vals, par, exp in examples + edge:
    for name, f in impls.items():
        got = f(vals, par)
        assert got == exp % MOD, (name, vals, par, got, exp)
    assert good_brute(vals, par) == exp % MOD
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [15]:
import random
random.seed(3575)

def rand_val():
    r = random.random()
    if r < 0.4:
        return random.randint(1, 9)
    if r < 0.8:
        return random.randint(10, 999)
    return random.randint(1, 10**9)

cnt = 0
for _ in range(400):
    n = random.randint(1, 11)
    par = [-1] + [random.randrange(i) for i in range(1, n)]
    vals = [rand_val() for _ in range(n)]
    exp = good_brute(vals, par)
    for name, f in impls.items():
        assert f(vals, par) == exp, (name, vals, par)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'cây ngẫu nhiên.')

Khớp vét cạn trên 400 cây ngẫu nhiên.


### 12.3. Ca lớn (`n = 500`): giá trị nhiều mask khác nhau (khó nhất cho DP)

In [16]:
import time

def distinct_digit_value():
    k = random.randint(1, 3)
    s = ''.join(random.sample('123456789', 1)) + ''.join(random.sample('0123456789', 10))[:k - 1]
    return int(s) if digit_mask(int(s)) > 0 else int(s[0])

random.seed(500)
n = 500
for name, par in [('ngẫu nhiên', [-1] + [random.randrange(i) for i in range(1, n)]),
                  ('đường thẳng', [-1] + list(range(n - 1))),
                  ('sao', [-1] + [0] * (n - 1))]:
    vals = [distinct_digit_value() for _ in range(n)]
    t = time.time(); a = Solution().goodSubtreeSum(vals, par); t1 = time.time() - t
    t = time.time(); b = good_convolution(vals, par); t2 = time.time() - t
    assert a == b
    print(f"{name:>11}: đáp án {a} | gộp nhỏ vào lớn {t1:.2f}s | tích chập {t2:.2f}s")

 ngẫu nhiên: đáp án 215155 | gộp nhỏ vào lớn 0.08s | tích chập 0.30s
đường thẳng: đáp án 1143748 | gộp nhỏ vào lớn 0.02s | tích chập 0.11s


        sao: đáp án 89155 | gộp nhỏ vào lớn 0.04s | tích chập 1.29s


Lời giải chính chạy dưới ~0,5 s; cách tích chập chậm hơn rõ khi mask dày đặc.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không loại số có chữ số lặp (chỉ kiểm tra giữa các số)

In [17]:
def mask_wrong(v):
    m = 0
    for ch in str(v):
        m |= 1 << (ord(ch) - 48)      # SAI: không phát hiện chữ số lặp trong chính v
    return m

def good_wrong1(vals, par):
    n = len(vals)
    total = 0
    for u in range(n):
        g = [0] * 1024
        children = [[] for _ in range(n)]
        for i in range(1, n):
            children[par[i]].append(i)
        st = [u]
        while st:
            x = st.pop()
            add_item(g, mask_wrong(vals[x]), vals[x])
            st.extend(children[x])
        total += g[FULL]
    return total % MOD

print('sai :', good_wrong1([3, 22, 5], [-1, 0, 1]))   # chọn được 22
print('đúng:', Solution().goodSubtreeSum([3, 22, 5], [-1, 0, 1]))

sai : 62
đúng: 18


**Vì sao sai:** `22` tự nó đã có chữ số 2 xuất hiện hai lần ⇒ tập chứa `22` không bao giờ tốt.

### Lỗi 2 — Tham lam: lấy giá trị lớn trước nếu không xung đột

In [18]:
def good_greedy(vals, par):
    n = len(vals)
    total = 0
    for u in range(n):
        used, s = 0, 0
        for m, v in sorted(items_of_subtree(vals, par, u), key=lambda t: -t[1]):
            if not used & m:
                used |= m
                s += v
        total += s
    return total % MOD

Tham lam "giá trị lớn trước" nghe hợp lý nhưng sai. Tìm phản ví dụ tự động (đối chiếu vét cạn):

In [19]:
random.seed(21)
found = None
for _ in range(5000):
    n = random.randint(2, 6)
    par = [-1] + [random.randrange(i) for i in range(1, n)]
    vals = [random.choice([random.randint(1, 9), random.randint(10, 99), random.randint(100, 999)]) for _ in range(n)]
    if good_greedy(vals, par) != good_brute(vals, par):
        found = (vals, par, good_greedy(vals, par), good_brute(vals, par))
        break
print('Phản ví dụ (vals, par, tham lam, đúng):', found)

Phản ví dụ (vals, par, tham lam, đúng): ([817, 98, 476, 795, 351], [-1, 0, 1, 2, 3], 3553, 3823)


**Vì sao sai:** đây là bài đóng gói tập (NP-khó tổng quát); một số lớn "ăn" nhiều chữ số có thể chặn nhiều số nhỏ mà tổng của chúng lớn hơn. Cần DP trên mask.

### Lỗi 3 — Knapsack cập nhật "tại chỗ" theo thứ tự `S` tăng dần với công thức `g[S] = max(g[S], v + g[S ^ m])` cho mọi `S` (kể cả `S` không chứa `m`)

In [20]:
def add_item_wrong(g, m, v):
    for S in range(1024):                         # SAI: không kiểm tra S chứa m
        T = S ^ m                                 # khi S không chứa m, T lại CHỨA m
        g[S] = max(g[S], v + g[T])

g_ok, g_bad = [0] * 1024, [0] * 1024
for v in [5, 3]:
    add_item(g_ok, digit_mask(v), v)
    add_item_wrong(g_bad, digit_mask(v), v)
print('sai :', g_bad[FULL], '(dùng số 5 hai lần?)')
print('đúng:', g_ok[FULL])

sai : 16 (dùng số 5 hai lần?)
đúng: 8


**Vì sao sai:** khi `S` không chứa `m`, `S ^ m` **thêm** bit của `m` thay vì bỏ đi; vật có thể bị dùng lặp và tạo tập không hợp lệ. Chỉ cập nhật `S ⊇ m` và tra `S \ m`.

### Lỗi 4 — Lấy modulo sau mỗi bước so sánh (so sánh số đã mod)

In [21]:
x, y = 10**9 + 8, 5            # x thật sự lớn hơn
print('so sánh sau mod:', x % MOD > y % MOD, '| so sánh thật:', x > y)

so sánh sau mod: False | so sánh thật: True


**Vì sao sai:** lấy `% MOD` trước khi so sánh `max` làm hỏng thứ tự. Ở bài này các giá trị đủ nhỏ để Python tính chính xác; chỉ lấy modulo cho **tổng cuối cùng**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Mỗi chữ số / chữ cái xuất hiện tối đa một lần" ⇒ **bitmask** trên vũ trụ nhỏ (10 hoặc 26).
- Chọn tập con các vật có mask rời nhau, tối đa tổng ⇒ **knapsack trên mask**: thêm vật bằng duyệt tập con phần bù.
- "Tính cho **mọi** cây con" ⇒ DP từ lá lên; tập vật của cha chứa tập vật của con ⇒ **kế thừa** + gộp nhỏ vào lớn.
- Gộp hai bảng DP mask rời nhau ⇒ tích chập max-plus `O(3^D)`.
- Thứ tự BFS đi ngược thay cho đệ quy.

## 15. Biến thể của bài

1. Mỗi chữ số được xuất hiện **tối đa hai lần** ⇒ trạng thái cơ số 3 (`3^10 = 59 049`).
2. Thay chữ số bằng **chữ cái** của chuỗi (LC 1239 *Maximum Length of a Concatenated String with Unique Characters*) — vũ trụ 26 phần tử, chỉ một tập (không có cây).
3. Hỏi **số lượng** tập tốt thay vì tổng lớn nhất ⇒ knapsack đếm (cộng thay vì max).
4. Đường đi thay vì cây con ⇒ khó hơn, cần cấu trúc khác.

In [22]:
# Biến thể 2: LC 1239 — độ dài lớn nhất khi nối các chuỗi có ký tự đôi một khác nhau
def max_length(arr):
    g = {0: 0}
    for w in arr:
        m = 0
        ok = True
        for ch in w:
            b = 1 << (ord(ch) - 97)
            if m & b:
                ok = False
                break
            m |= b
        if not ok:
            continue
        for S, L in list(g.items()):
            if not S & m:
                g[S | m] = max(g.get(S | m, 0), L + len(w))
    return max(g.values())

print(max_length(["un", "iq", "ue"]), max_length(["cha", "r", "act", "ers"]))

4 6


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | Vét cạn tập con | mũ | `O(n)` | Oracle `n ≤ 12` |
| 2 | Knapsack riêng mỗi cây con | `O(n^2 · 2^D)` | `O(2^D)` | Dễ viết, `n` nhỏ |
| 3 | Tích chập max-plus | `O(n · 3^D)` | `O(h 2^D)` | Cách "chuẩn" theo gợi ý |
| 4 | Gộp nhỏ vào lớn | `O(n log n · 2^D)` | `O(n 2^D)` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `goodSubtreeSum([10, 2, 3], [-1, 0, 0])` = ?

**Bài 2 (tính tay):** `vals=[123, 45, 1, 6], par=[-1, 0, 1, 1]` — tính `maxScore` từng đỉnh.

**Bài 3 (điền TODO):** viết `digit_mask_todo(v)`.

**Bài 4 (tự viết lại):** biến thể 3 — đếm số tập tốt (kể cả tập rỗng) trong toàn cây.

**Bài 5 (mini challenge):** trả về **tập đỉnh** đạt `maxScore[0]`.

In [23]:
def digit_mask_todo(v):
    # TODO: bật bit cho từng chữ số, trả -1 nếu lặp
    raise NotImplementedError

def count_good_sets_todo(vals):
    # TODO: knapsack đếm theo mask
    raise NotImplementedError

def best_set_todo(vals, par):
    # TODO: knapsack có truy vết
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Ba số dùng chữ số {1,0}, {2}, {3} — rời nhau. (2) Gốc: 15; lá: 2 và 3. (3) Tổng 20.

**Bài 2.** (1) Đỉnh 2: 1; đỉnh 3: 6. (2) Đỉnh 1: {45, 1, 6} rời nhau ⇒ 52. (3) Gốc: so `123 + 45 + 6 = 174` với `45 + 1 + 6 = 52` ⇒ 174. Tổng `1 + 6 + 52 + 174 = 233`.

**Bài 3.** (1) `str(v)`. (2) `1 << digit`. (3) Kiểm tra `m & b` trước khi bật.

**Bài 4.** (1) `cnt[0] = 1`. (2) Thêm vật: `cnt[m | sub] += cnt[sub]` với `sub ⊆ ~m` (dùng bản sao cũ cho an toàn). (3) Tổng mọi `cnt[S]`.

**Bài 5.** (1) Lưu `who[S]` = danh sách vật tạo ra `g[S]`. (2) Khi cập nhật `g[m | sub]`, `who[m|sub] = who[sub] + [vật]`. (3) Kiểm tra tổng và tính rời nhau.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
a1 = Solution().goodSubtreeSum([10, 2, 3], [-1, 0, 0])
a2 = Solution().goodSubtreeSum([123, 45, 1, 6], [-1, 0, 1, 1])
assert a1 == 20 == good_brute([10, 2, 3], [-1, 0, 0])
assert a2 == 233 == good_brute([123, 45, 1, 6], [-1, 0, 1, 1])
print('Bài 1:', a1, '| Bài 2:', a2)

Bài 1: 20 | Bài 2: 233


In [25]:
# Bài 3
def digit_mask_ans(v):
    m = 0
    for ch in str(v):
        b = 1 << int(ch)
        if m & b:
            return -1
        m |= b
    return m

for v in list(range(1, 2000)) + [1023456789, 1000000000, 987654321]:
    s = str(v)
    exp = -1 if len(set(s)) < len(s) else sum(1 << int(c) for c in s)
    assert digit_mask_ans(v) == exp
print('Bài 3: đúng trên', 2002, 'giá trị')

Bài 3: đúng trên 2002 giá trị


In [26]:
# Bài 4: đếm số tập tốt trong toàn cây (kể cả rỗng)
def count_good_sets(vals):
    cnt = [0] * 1024
    cnt[0] = 1
    for v in vals:
        m = digit_mask(v)
        if m < 0:
            continue
        old = cnt[:]
        comp = FULL ^ m
        sub = comp
        while True:
            cnt[m | sub] += old[sub]
            if sub == 0:
                break
            sub = (sub - 1) & comp
    return sum(cnt)

def count_brute(vals):
    k, c = len(vals), 0
    for pick in range(1 << k):
        used, ok = 0, True
        for i in range(k):
            if pick >> i & 1:
                m = digit_mask(vals[i])
                if m < 0 or used & m:
                    ok = False
                    break
                used |= m
        c += ok
    return c

random.seed(44)
for _ in range(200):
    vals = [rand_val() for _ in range(random.randint(1, 9))]
    assert count_good_sets(vals) == count_brute(vals)
print('Bài 4: ví dụ', count_good_sets([1, 2, 12, 3]))

Bài 4: ví dụ 10


In [27]:
# Bài 5: truy vết tập đỉnh tốt nhất ở gốc
def best_set(vals, par):
    g = [0] * 1024
    who = [[] for _ in range(1024)]
    for u in range(len(vals)):            # cây con của gốc = mọi đỉnh
        m = digit_mask(vals[u])
        if m < 0:
            continue
        comp = FULL ^ m
        sub = comp
        while True:
            if vals[u] + g[sub] > g[m | sub]:
                g[m | sub] = vals[u] + g[sub]
                who[m | sub] = who[sub] + [u]
            if sub == 0:
                break
            sub = (sub - 1) & comp
    return g[FULL], who[FULL]

score, nodes = best_set(vals6, par6)
used = 0
for u in nodes:
    assert used & digit_mask(vals6[u]) == 0
    used |= digit_mask(vals6[u])
assert score == sum(vals6[u] for u in nodes) == 100
print('Bài 5: các đỉnh', nodes, 'giá trị', [vals6[u] for u in nodes], 'tổng', score)

Bài 5: các đỉnh [0, 1, 2, 3, 5] giá trị [12, 30, 45, 7, 6] tổng 100


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
mask(v): bật bit từng chữ số, lặp -> -1 (bỏ)
add(g, m, v): với sub ⊆ ~m: g[m|sub] = max(g[m|sub], v + g[sub])
BFS order; đi ngược:
    g, items = của con có nhiều vật nhất (hoặc mới)
    thêm vật của các con khác + của u
    maxScore[u] = g[1023]
return Σ maxScore % MOD
```

**Checklist:**

- [ ] Loại số có chữ số lặp (kể cả `0` lặp, ví dụ `100`)?
- [ ] Knapsack chỉ cập nhật `S ⊇ m` và tra `S \ m`?
- [ ] Kế thừa mảng của con lớn nhất, **không** sao chép?
- [ ] Cộng `maxScore[c]` trước khi mảng bị cha tái sử dụng?
- [ ] Modulo chỉ ở cuối?